# Inferring a Planetary Field from a Radio Cutoff

This example uses Aegis's cyclotron-cutoff relation and `infer_surface_field()` to infer a surface field from two explicitly synthetic cutoff measurements. Each cutoff is generated from a stated assumed field, then perturbed by a seeded Gaussian measurement error. The notebook plots the two PCAT posterior distributions and reports their central 68% intervals.

The observations, assumed fields, and uncertainties are toy inputs, not telescope measurements. The short chains illustrate the API and prior/likelihood flow; they do not establish convergence or constrain any real exoplanet.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
repository_root = next(
    candidate for candidate in search_roots
    if (candidate / "magnetic_field_inference.py").is_file()
    and (candidate / "radio_module.py").is_file()
)
for import_root in (
    repository_root,
    repository_root.parent.parent / "tansudaylan" / "tdpy",
):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

from magnetic_field_inference import infer_surface_field
from radio_module import max_freq
from tdpy.verbosity import print

visuals_directory = repository_root / "visuals"
visuals_directory.mkdir(exist_ok=True)
data_directory = repository_root / "data" / "magnetic_field_notebook"
data_directory.mkdir(exist_ok=True)
rng = np.random.default_rng(2026)
measurement_uncertainty_mhz = 2.0  # [MHz]
assumed_fields_gauss = np.array([18.0, 36.0])  # [G]

In [ ]:
posterior_results = []
case_rows = []

for case_index, assumed_field_gauss in enumerate(assumed_fields_gauss):
    predicted_cutoff_mhz = max_freq(assumed_field_gauss) / 1.0e6  # [MHz]
    observed_cutoff_mhz = float(
        rng.normal(predicted_cutoff_mhz, measurement_uncertainty_mhz)
    )  # [MHz]
    case_path = data_directory / f"case_{case_index + 1}"
    posterior = infer_surface_field(
        observed_cutoff_mhz,
        measurement_uncertainty_mhz,
        field_bounds_gauss=(1.0, 100.0),  # [G]
        chain_count=2,
        sample_count=160,
        burn_count=80,
        seed=case_index + 10,
        pathbase=str(case_path),
    )
    posterior_results.append(posterior)
    lower_gauss, median_gauss, upper_gauss = posterior.field_interval_gauss
    case_rows.append({
        "assumed_field_gauss": assumed_field_gauss,  # [G]
        "predicted_cutoff_mhz": predicted_cutoff_mhz,  # [MHz]
        "observed_cutoff_mhz": observed_cutoff_mhz,  # [MHz]
        "lower_68_gauss": lower_gauss,  # [G]
        "median_gauss": median_gauss,  # [G]
        "upper_68_gauss": upper_gauss,  # [G]
    })

case_table = pd.DataFrame(case_rows)
display(case_table.round(2))

In [ ]:
figure, axes = plt.subplots(1, len(posterior_results), figsize=(10, 3.8), constrained_layout=True)
for case_index, (axis, posterior, assumed_field_gauss) in enumerate(
    zip(axes, posterior_results, assumed_fields_gauss),
    start=1,
):
    field_samples_gauss = posterior.chain_gauss.ravel()
    lower_gauss, median_gauss, upper_gauss = posterior.field_interval_gauss
    axis.hist(field_samples_gauss, bins=24, density=True, color="#007360", alpha=0.78)
    axis.axvline(assumed_field_gauss, color="#A51417", linestyle="--", label="Assumed toy field")
    axis.axvline(median_gauss, color="#222222", linewidth=1.1, label="Posterior median")
    axis.set_title(f"Synthetic cutoff case {case_index}")
    axis.set_xlabel("Planetary surface field [G]")
    axis.set_ylabel("Posterior density [G$^{-1}$]")
    axis.grid(False)
    axis.legend(loc="upper right", fontsize=8)
figure.patch.set_facecolor("white")

figure_path = visuals_directory / "aegis_synthetic_cutoff_field_posterior.png"
print(f"Writing to visuals/{figure_path.name}...")
figure.savefig(figure_path, dpi=300, bbox_inches="tight", facecolor="white")
display(figure)
plt.close(figure)